In [6]:

# CNTNAP2 LFP PROCESSING — Initialization

%pip -q install -U ONE-api ibllib iblatlas scipy

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import defaultdict
from one.api import ONE
from iblatlas.regions import BrainRegions
from brainbox.io.one import SpikeSortingLoader
from scipy.signal import welch

import numpy as np
import pandas as pd
import hashlib
import gc

one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    username="intbrainlab",
    password="international",
    silent=True
)

br = BrainRegions()

GENOTYPE = "Cntnap2"
PROJECT = "angelaki_mouseASD"

ROOT = Path(
    "/content/drive/MyDrive/Reanalysis/"
    "Our_Analysis/lfp/spontaneous/Cntnap2/processing"
)

ROOT.mkdir(parents=True, exist_ok=True)

INSERTION_DIR = ROOT / "per_insertion"
INSERTION_DIR.mkdir(exist_ok=True)

N_WINDOWS = 30
WINDOW_S = 2.0
GUARD_S = 0.5
TUKEY_K = 3.0
MIN_ANIMALS = 5

BANDS = {
    "Delta": (1, 4),
    "Theta": (4, 8),
    "Alpha-like": (8, 12),
    "Beta": (12, 30),
    "Low gamma": (30, 55),
    "High gamma": (65, 100)
}

REGIONS = [
    "DG", "CA1", "CA3", "SUB",
    "VISp", "MOs", "CP", "MRN", "APN"
]

print("Genotype:", GENOTYPE)
print("Output:", ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Genotype: Cntnap2
Output: /content/drive/MyDrive/Reanalysis/Our_Analysis/lfp/spontaneous/Cntnap2/processing


In [ ]:

# ============================================================
# STEP 2 — CNTNAP2 INVENTORY (CORRECTED)
# ============================================================

GENOTYPE = "Cntnap2"
PROJECT = "angelaki_mouseASD"

# In this IBL dataset, Cntnap2 subjects use the CSP prefix.
subjects = one.alyx.rest("subjects", "list")

candidate_subjects = sorted({
    str(s["nickname"])
    for s in subjects
    if str(s.get("nickname", "")).upper().startswith("CSP")
})

print("CSP candidate subjects:", len(candidate_subjects))
print(candidate_subjects)

inventory_rows = []
project_subjects = set()

for i, subject in enumerate(candidate_subjects, 1):

    eids = one.search(
        project=PROJECT,
        subject=subject
    )

    if not len(eids):
        continue

    project_subjects.add(subject)

    print(
        f"[{i}/{len(candidate_subjects)}] "
        f"{subject}: {len(eids)} sessions"
    )

    for eid_i in eids:
        eid_i = str(eid_i)

        info = one.alyx.rest(
            "sessions", "read", id=eid_i
        )

        if str(info.get("subject")) != subject:
            continue

        insertions = one.alyx.rest(
            "insertions",
            "list",
            session=eid_i
        )

        paths = [
            str(p) for p in one.list_datasets(eid_i)
        ]

        for ins in insertions:
            probe = str(ins["name"])

            lf_files = [
                p for p in paths
                if p.startswith(
                    f"raw_ephys_data/{probe}/"
                )
                and p.endswith(".lf.cbin")
            ]

            inventory_rows.append({
                "subject": subject,
                "genotype": GENOTYPE,
                "eid": eid_i,
                "pid": str(ins["id"]),
                "probe": probe,
                "date": info.get("start_time"),
                "lf_available": bool(lf_files),
                "lf_files": len(lf_files)
            })

inventory_all = pd.DataFrame(inventory_rows)

if inventory_all.empty:
    raise RuntimeError(
        "No CSP insertions found in the project."
    )

inventory = (
    inventory_all.loc[
        inventory_all["lf_available"]
    ]
    .drop_duplicates("pid")
    .reset_index(drop=True)
)

# Preserve full and LF-only inventories.
inventory_all.to_csv(
    ROOT / "Cntnap2_LFP_inventory_all.csv",
    index=False
)

inventory.to_csv(
    ROOT / "Cntnap2_LFP_inventory.csv",
    index=False
)

print("\n" + "=" * 50)
print("CNTNAP2 INVENTORY")
print("=" * 50)

print("Project CSP subjects:", len(project_subjects))
print("Animals with LF:", inventory["subject"].nunique())
print("Sessions:", inventory["eid"].nunique())
print("LF insertions:", len(inventory))

display(inventory.head(10))

# Verify the LF cohort, not all CSP subjects in Alyx.
n_animals = inventory["subject"].nunique()
n_insertions = len(inventory)

print(f"Cntnap2 animals with LF: {n_animals}")
print(f"LF insertions: {n_insertions}")

if n_animals == 0 or n_insertions == 0:
    raise RuntimeError("No Cntnap2 LF data available.")

print("Inventory ready for processing.")


CSP candidate subjects: 25
['CSP001', 'CSP003', 'CSP004', 'CSP005', 'CSP006', 'CSP009', 'CSP010', 'CSP011', 'CSP012', 'CSP013', 'CSP015', 'CSP016', 'CSP017', 'CSP018', 'CSP022', 'CSP023', 'CSP025', 'CSP026', 'CSP027', 'CSP028', 'CSP029', 'CSP030', 'CSP031', 'CSP032', 'CSP033']
[1/25] CSP001: 75 sessions
[2/25] CSP003: 71 sessions
[3/25] CSP004: 46 sessions
[4/25] CSP005: 54 sessions
[5/25] CSP006: 51 sessions
[6/25] CSP009: 73 sessions
[7/25] CSP010: 14 sessions
[8/25] CSP011: 74 sessions
[9/25] CSP012: 80 sessions
[10/25] CSP013: 100 sessions
[11/25] CSP015: 95 sessions
[12/25] CSP016: 35 sessions
[13/25] CSP017: 54 sessions
[14/25] CSP018: 95 sessions


In [ ]:

# ============================================================
# CELL 3 — CNTNAP2 LFP PROCESSING FUNCTIONS
# Identical analysis strategy to WT and Fmr1
# ============================================================

import numpy as np
import pandas as pd
import hashlib

from pathlib import Path
from collections import defaultdict
from scipy.signal import welch
from iblatlas.regions import BrainRegions

br = BrainRegions()

GENOTYPE = "Cntnap2"

REGIONS = [
    "DG", "CA1", "CA3", "SUB",
    "VISp", "MOs", "CP", "MRN", "APN"
]

BANDS = {
    "Delta": (1, 4),
    "Theta": (4, 8),
    "Alpha-like": (8, 12),
    "Beta": (12, 30),
    "Low gamma": (30, 55),
    "High gamma": (65, 100)
}

N_WINDOWS = 30
WINDOW_S = 2.0
GUARD_S = 0.5
TUKEY_K = 3.0
MIN_ANIMALS = 5

ROOT = Path(
    "/content/drive/MyDrive/Reanalysis/"
    "Our_Analysis/lfp/spontaneous/Cntnap2/processing"
)

INSERTION_DIR = ROOT / "per_insertion"
ROOT.mkdir(parents=True, exist_ok=True)
INSERTION_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1 — Identify anatomical regions and subregions
# ------------------------------------------------------------

def classify_region(acronym):
    a = str(acronym)

    if a in ("DG-mo", "DG-sg", "DG-po"):
        return "DG", a

    for region in REGIONS:

        if a == region:
            return region, a

        if region in ("CA1", "CA3", "SUB", "VISp", "MOs"):
            if a.startswith(region):
                suffix = a[len(region):]

                if suffix and (
                    suffix[0].isdigit()
                    or suffix[0] == "-"
                ):
                    return region, a

    return None, None


# ------------------------------------------------------------
# 2 — Recover anatomical channel mappings
# ------------------------------------------------------------

def get_channel_groups(eid, probe):
    """
    Search for anatomical annotations in all ALF collections
    belonging to the probe, including dated revisions.

    Priority:
      channels + rawInd
      electrodeSites + verified coordinate mapping
    """

    paths = [str(p) for p in one.list_datasets(eid)]
    prefix = f"alf/{probe}/"

    catalog = defaultdict(set)

    for path in paths:

        if not path.startswith(prefix):
            continue

        parts = path.split("/")
        filename = parts[-1]

        if not filename.startswith(
            ("channels.", "electrodeSites.")
        ):
            continue

        revision = None

        if (
            len(parts) >= 3
            and parts[-2].startswith("#")
            and parts[-2].endswith("#")
        ):
            revision = parts[-2][1:-1]
            collection = "/".join(parts[:-2])
        else:
            collection = "/".join(parts[:-1])

        obj = filename.split(".")[0]

        catalog[
            (collection, revision, obj)
        ].add(filename)

    def load_attr(collection, revision, obj, attr):

        kwargs = {"collection": collection}

        if revision is not None:
            kwargs["revision"] = revision

        return np.asarray(
            one.load_dataset(
                eid,
                f"{obj}.{attr}.npy",
                **kwargs
            )
        )

    candidates = []

    for (col, rev, obj), names in catalog.items():

        if (
            f"{obj}.brainLocationIds_ccf_2017.npy"
            in names
        ):
            candidates.append(
                (col, rev, obj, names)
            )

    # Prefer channels with an explicit raw mapping.
    candidates.sort(
        key=lambda x: (
            x[2] != "channels",
            x[1] is None
        )
    )

    recovered = []
    errors = []

    for col, rev, obj, names in candidates:

        try:
            atlas = load_attr(
                col, rev, obj,
                "brainLocationIds_ccf_2017"
            ).ravel().astype(int)

            raw = None
            mapping = None

            # Option A: Direct raw channel mapping.
            if f"{obj}.rawInd.npy" in names:

                raw = load_attr(
                    col, rev, obj, "rawInd"
                ).ravel().astype(int)

                if len(raw) != len(atlas):
                    raise ValueError(
                        "Atlas/rawInd length mismatch"
                    )

                mapping = "explicit_rawInd"

            # Option B: Match electrode site coordinates.
            elif (
                obj == "electrodeSites"
                and "electrodeSites.localCoordinates.npy"
                in names
            ):

                site_xy = load_attr(
                    col, rev, obj,
                    "localCoordinates"
                )

                for (cc, rr, oo), other in catalog.items():

                    if oo != "channels":
                        continue

                    required = {
                        "channels.rawInd.npy",
                        "channels.localCoordinates.npy"
                    }

                    if not required.issubset(other):
                        continue

                    try:
                        ch_raw = load_attr(
                            cc, rr, "channels", "rawInd"
                        ).ravel().astype(int)

                        ch_xy = load_attr(
                            cc, rr, "channels",
                            "localCoordinates"
                        )

                        if len(ch_raw) != len(ch_xy):
                            continue

                        def xy_key(x):
                            return tuple(
                                np.round(
                                    np.asarray(x)[:2], 3
                                )
                            )

                        xy_lookup = defaultdict(list)

                        for xy, channel in zip(
                            ch_xy, ch_raw
                        ):
                            xy_lookup[
                                xy_key(xy)
                            ].append(int(channel))

                        raw_match = []
                        atlas_match = []

                        for xy, atlas_id in zip(
                            site_xy, atlas
                        ):

                            matches = xy_lookup.get(
                                xy_key(xy), []
                            )

                            if len(matches) == 1:
                                raw_match.append(
                                    matches[0]
                                )
                                atlas_match.append(
                                    int(atlas_id)
                                )

                        if raw_match:
                            raw = np.asarray(
                                raw_match, dtype=int
                            )

                            atlas = np.asarray(
                                atlas_match, dtype=int
                            )

                            mapping = "coordinate_match"
                            break

                    except Exception:
                        continue

            if raw is None:
                raise ValueError(
                    "No verified raw-channel mapping"
                )

            if len(np.unique(raw)) != len(raw):
                raise ValueError(
                    "Duplicate raw channel indices"
                )

            valid = (
                (raw >= 0)
                & (raw < 384)
            )

            raw = raw[valid]
            atlas = atlas[valid]

            acronyms = br.id2acronym(atlas)

            groups = defaultdict(list)

            for channel, acronym in zip(raw, acronyms):

                region, subregion = classify_region(
                    acronym
                )

                if region is None:
                    continue

                groups[
                    (region, "ALL")
                ].append(int(channel))

                groups[
                    (region, subregion)
                ].append(int(channel))

            if not groups:
                raise ValueError(
                    "No target anatomical regions"
                )

            groups = {
                key: np.unique(values).astype(int)
                for key, values in groups.items()
            }

            n_contacts = len(set(
                channel
                for channels in groups.values()
                for channel in channels
            ))

            recovered.append({
                "groups": groups,
                "n_contacts": n_contacts,
                "mapping": mapping,
                "collection": col,
                "revision": rev
            })

        except Exception as exc:
            errors.append(str(exc))

    if not recovered:
        raise RuntimeError(
            "Anatomy unavailable: "
            + "; ".join(errors[:3])
        )

    # Select mapping with maximum anatomical coverage.
    best = max(
        recovered,
        key=lambda x: x["n_contacts"]
    )

    return best["groups"]


# ------------------------------------------------------------
# 3 — Select reproducible quiet windows
# ------------------------------------------------------------

def get_quiet_windows(eid):

    trials = pd.DataFrame(
        one.load_dataset(
            eid,
            "_ibl_trials.table.pqt",
            collection="alf"
        )
    ).sort_values(
        "intervals_0"
    ).reset_index(drop=True)

    movements = np.asarray(
        one.load_dataset(
            eid,
            "_ibl_wheelMoves.intervals.npy",
            collection="alf"
        ),
        dtype=float
    ).reshape(-1, 2)

    movements = movements[
        np.isfinite(movements).all(axis=1)
    ]

    eligible = []

    for i in range(len(trials) - 1):

        start = (
            float(trials.loc[i, "feedback_times"])
            + GUARD_S
        )

        end = (
            float(trials.loc[i + 1, "stimOn_times"])
            - GUARD_S
        )

        if (
            not np.isfinite(start)
            or not np.isfinite(end)
        ):
            continue

        if end - start < WINDOW_S:
            continue

        segments = [(start, end)]

        overlapping = movements[
            (movements[:, 0] < end)
            & (movements[:, 1] > start)
        ]

        for m0, m1 in overlapping:

            updated = []

            for a, b in segments:

                if m1 <= a or m0 >= b:
                    updated.append((a, b))

                else:
                    if m0 > a:
                        updated.append(
                            (a, float(m0))
                        )

                    if m1 < b:
                        updated.append(
                            (float(m1), b)
                        )

            segments = updated

        eligible.extend(
            (a, b)
            for a, b in segments
            if b - a >= WINDOW_S
        )

    # Stable random seed based on session UUID.
    seed = int(
        hashlib.md5(
            str(eid).encode()
        ).hexdigest()[:8],
        16
    )

    rng = np.random.default_rng(seed)

    windows = []

    for idx in rng.permutation(len(eligible)):

        a, b = eligible[idx]

        t0 = float(
            rng.uniform(a, b - WINDOW_S)
        )

        windows.append(
            (t0, t0 + WINDOW_S)
        )

        if len(windows) == N_WINDOWS:
            break

    return np.asarray(
        sorted(windows),
        dtype=float
    ).reshape(-1, 2), len(eligible)


# ------------------------------------------------------------
# 4 — Tukey artifact rejection
# ------------------------------------------------------------

def tukey_keep(values):

    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)

    if not finite.any():
        return np.zeros(
            len(values), dtype=bool
        )

    q1, q3 = np.percentile(
        values[finite],
        [25, 75]
    )

    threshold = q3 + TUKEY_K * (q3 - q1)

    return finite & (values <= threshold)


# ------------------------------------------------------------
# 5 — Integrate PSD into frequency bands
# ------------------------------------------------------------

def integrate_power(
    psd, frequencies, low, high
):

    mask = (
        (frequencies >= low)
        & (frequencies <= high)
    )

    return np.trapezoid(
        psd[..., mask],
        frequencies[mask],
        axis=-1
    )


# ------------------------------------------------------------
# 6 — Aggregate power for one anatomical group
# ------------------------------------------------------------

def calculate_group_power(
    psd, rms, frequencies, indices
):
    """
    psd: windows x channels x frequencies
    rms: windows x channels

    Returns insertion-level band powers.
    """

    subset_psd = psd[:, indices, :]
    subset_rms = rms[:, indices]

    # Group-level window QC.
    window_rms = np.median(
        subset_rms, axis=1
    )

    window_keep = tukey_keep(window_rms)

    if not window_keep.any():
        return None

    # Individual channel/window QC.
    valid = np.zeros_like(
        subset_rms,
        dtype=bool
    )

    for j in range(subset_rms.shape[1]):

        r = subset_rms[window_keep, j]

        if len(r) >= 4:
            valid[window_keep, j] = (
                tukey_keep(r)
            )
        else:
            valid[window_keep, j] = (
                np.isfinite(r)
            )

    powers = {
        name: integrate_power(
            subset_psd, frequencies, low, high
        )
        for name, (low, high) in BANDS.items()
    }

    # Total power excludes the 55–65 Hz interval.
    total = (
        integrate_power(
            subset_psd, frequencies, 1, 55
        )
        + integrate_power(
            subset_psd, frequencies, 65, 100
        )
    )

    contact_abs = defaultdict(list)
    contact_rel = defaultdict(list)
    n_good_channels = 0

    for j in range(subset_rms.shape[1]):

        good = valid[:, j]

        if not good.any():
            continue

        n_good_channels += 1

        for name in BANDS:

            absolute = float(
                np.mean(
                    powers[name][good, j]
                )
            )

            relative = float(
                np.mean(
                    100 * powers[name][good, j]
                    / np.maximum(
                        total[good, j], 1e-20
                    )
                )
            )

            contact_abs[name].append(
                absolute
            )

            contact_rel[name].append(
                relative
            )

    if n_good_channels == 0:
        return None

    result = {
        "n_channels": n_good_channels,
        "n_windows_available": len(psd),
        "n_windows_qc": int(window_keep.sum()),
        "n_channel_windows_qc": int(
            valid.sum()
        )
    }

    # Median across anatomical contacts.
    for name in BANDS:

        result[
            f"{name}_absolute_uv2"
        ] = float(
            np.median(contact_abs[name])
        )

        result[
            f"{name}_relative_pct"
        ] = float(
            np.median(contact_rel[name])
        )

    return result


print("=" * 55)
print("CNTNAP2 CORE FUNCTIONS READY")
print("=" * 55)
print("Regions:", len(REGIONS))
print("Bands:", len(BANDS))
print("Windows:", N_WINDOWS, "x", WINDOW_S, "seconds")
print("Tukey coefficient:", TUKEY_K)
print("Minimum animals:", MIN_ANIMALS)


In [ ]:

# ============================================================
# CELL 4 — CNTNAP2 BATCH LFP PROCESSING
# ============================================================

import gc
from brainbox.io.one import SpikeSortingLoader
from scipy.signal import welch

GENOTYPE = "Cntnap2"

LOG_PATH = (
    ROOT / "Cntnap2_LFP_processing_log.csv"
)

# Restore processing history when restarting Colab.
if LOG_PATH.exists():
    previous_log = pd.read_csv(LOG_PATH)

    log_by_pid = {
        str(row["pid"]): dict(row)
        for _, row in previous_log.iterrows()
    }
else:
    log_by_pid = {}

session_window_cache = {}

for number, row in enumerate(
    inventory.itertuples(index=False), start=1
):

    subject = str(row.subject)
    eid_i = str(row.eid)
    pid_i = str(row.pid)
    probe = str(row.probe)

    output_path = (
        INSERTION_DIR / f"{pid_i}.csv"
    )

    windows_path = (
        INSERTION_DIR / f"{pid_i}_windows.csv"
    )

    # --------------------------------------------------------
    # Resume previously completed recordings
    # --------------------------------------------------------

    if output_path.exists() and windows_path.exists():

        try:
            cached = pd.read_csv(output_path)

            required = {
                "subject", "genotype", "eid", "pid",
                "probe", "region", "subregion",
                "n_channels", "n_windows_qc"
            }

            required.update(
                f"{band}_{metric}"
                for band in BANDS
                for metric in (
                    "absolute_uv2",
                    "relative_pct"
                )
            )

            if (
                not cached.empty
                and required.issubset(cached.columns)
                and cached["genotype"].eq(
                    GENOTYPE
                ).all()
                and cached["pid"].astype(str).eq(
                    pid_i
                ).all()
            ):

                n_cached_windows = len(
                    pd.read_csv(windows_path)
                )

                log_by_pid[pid_i] = {
                    "pid": pid_i,
                    "subject": subject,
                    "status": "processed",
                    "n_windows": n_cached_windows,
                    "n_regions": len(cached),
                    "error": ""
                }

                print(
                    f"[{number}/{len(inventory)}] "
                    f"CACHED {subject} {probe}"
                )

                continue

        except Exception:
            pass

    print(
        f"[{number}/{len(inventory)}] "
        f"{subject} — {probe}",
        flush=True
    )

    try:

        # ----------------------------------------------------
        # 1. Load anatomical channel groups
        # ----------------------------------------------------

        groups = get_channel_groups(
            eid_i, probe
        )

        if not groups:
            raise RuntimeError(
                "No anatomical groups available"
            )

        # ----------------------------------------------------
        # 2. Select quiet windows
        # ----------------------------------------------------

        if eid_i not in session_window_cache:

            session_window_cache[eid_i] = (
                get_quiet_windows(eid_i)
            )

        windows, n_eligible = (
            session_window_cache[eid_i]
        )

        if len(windows) == 0:
            raise RuntimeError(
                "No eligible 2-second quiet windows"
            )

        # ----------------------------------------------------
        # 3. Open raw LFP stream
        # ----------------------------------------------------

        ssl_i = SpikeSortingLoader(
            pid=pid_i,
            one=one
        )

        reader = ssl_i.raw_electrophysiology(
            band="lf",
            stream=True
        )

        fs_i = float(reader.fs)

        n_neural = int(
            reader.nc - reader.nsync
        )

        n_samples = int(
            round(WINDOW_S * fs_i)
        )

        # ----------------------------------------------------
        # 4. Select all target anatomical channels
        # ----------------------------------------------------

        selected_channels = np.unique(
            np.concatenate(
                list(groups.values())
            )
        )

        selected_channels = selected_channels[
            (selected_channels >= 0)
            & (selected_channels < n_neural)
        ]

        if len(selected_channels) == 0:
            raise RuntimeError(
                "No mapped LF channels"
            )

        local_lookup = {
            int(raw_channel): local_idx
            for local_idx, raw_channel in enumerate(
                selected_channels
            )
        }

        group_indices = {}

        for key, raw_channels in groups.items():

            local_indices = [
                local_lookup[int(ch)]
                for ch in raw_channels
                if int(ch) in local_lookup
            ]

            if local_indices:
                group_indices[key] = np.asarray(
                    local_indices, dtype=int
                )

        # ----------------------------------------------------
        # 5. Read matching LFP windows
        # ----------------------------------------------------

        psd_list = []
        rms_list = []
        loaded_windows = []
        window_errors = []

        for t0, t1 in windows:

            try:

                probe_t0 = float(
                    ssl_i.timesprobe2times(
                        float(t0),
                        direction="reverse"
                    )
                )

                s0 = int(round(probe_t0 * fs_i))
                s1 = s0 + n_samples

                if s0 < 0:
                    continue

                # Stream the full neural block once.
                raw_block = np.asarray(
                    reader[s0:s1, :n_neural]
                )

                if raw_block.shape != (
                    n_samples, n_neural
                ):
                    continue

                signals_uv = (
                    raw_block[
                        :, selected_channels
                    ].astype(np.float64)
                    * 1e6
                )

                if not np.isfinite(
                    signals_uv
                ).all():
                    continue

                # Welch PSD for every anatomical channel.
                frequencies, psd = welch(
                    signals_uv,
                    fs=fs_i,
                    window="hann",
                    nperseg=n_samples,
                    noverlap=0,
                    detrend="linear",
                    scaling="density",
                    axis=0
                )

                freq_mask = (
                    frequencies <= 100
                )

                # Shape: channels x frequencies
                psd_list.append(
                    psd[
                        freq_mask, :
                    ].T.astype(np.float32)
                )

                rms_list.append(
                    np.std(
                        signals_uv,
                        axis=0
                    )
                )

                loaded_windows.append(
                    (t0, t1)
                )

            except Exception as exc:

                window_errors.append(
                    str(exc)[:150]
                )

        if not psd_list:
            raise RuntimeError(
                "No LFP windows loaded: "
                + "; ".join(
                    window_errors[:3]
                )
            )

        # Shape: windows x channels x frequencies
        psd_array = np.stack(
            psd_list, axis=0
        )

        rms_array = np.stack(
            rms_list, axis=0
        )

        freqs_used = frequencies[
            freq_mask
        ]

        # ----------------------------------------------------
        # 6. Compute band power by anatomical group
        # ----------------------------------------------------

        results = []

        for (
            region, subregion
        ), indices in group_indices.items():

            metrics = calculate_group_power(
                psd_array,
                rms_array,
                freqs_used,
                indices
            )

            if metrics is None:
                continue

            results.append({
                "subject": subject,
                "genotype": GENOTYPE,
                "eid": eid_i,
                "pid": pid_i,
                "probe": probe,
                "region": region,
                "subregion": subregion,
                **metrics
            })

        if not results:
            raise RuntimeError(
                "No valid regional power estimates"
            )

        # ----------------------------------------------------
        # 7. Save insertion results and window times
        # ----------------------------------------------------

        pd.DataFrame(
            results
        ).to_csv(
            output_path,
            index=False
        )

        pd.DataFrame(
            loaded_windows,
            columns=["start_s", "end_s"]
        ).to_csv(
            windows_path,
            index=False
        )

        log_by_pid[pid_i] = {
            "pid": pid_i,
            "subject": subject,
            "status": "processed",
            "n_windows": len(loaded_windows),
            "n_regions": len(results),
            "error": ""
        }

        print(
            f"  OK | {len(loaded_windows)} windows | "
            f"{len(results)} anatomical groups"
        )

        del psd_array, rms_array
        del psd_list, rms_list

        gc.collect()

    except Exception as exc:

        # Remove incomplete results from unsuccessful processing.
        if output_path.exists():
            output_path.unlink()

        if windows_path.exists():
            windows_path.unlink()

        log_by_pid[pid_i] = {
            "pid": pid_i,
            "subject": subject,
            "status": "failed",
            "n_windows": 0,
            "n_regions": 0,
            "error": str(exc)
        }

        print(
            "  FAILED:",
            str(exc)[:200]
        )

    finally:

        # Save progress after every insertion.
        pd.DataFrame(
            list(log_by_pid.values())
        ).to_csv(
            LOG_PATH,
            index=False
        )

# ------------------------------------------------------------
# FINAL BATCH REPORT
# ------------------------------------------------------------

log_df = pd.DataFrame(
    list(log_by_pid.values())
)

print("\n" + "=" * 60)
print("CNTNAP2 BATCH PROCESSING COMPLETE")
print("=" * 60)

print(
    "Processed insertions:",
    int(
        (log_df["status"] == "processed").sum()
    )
)

print(
    "Failed insertions:",
    int(
        (log_df["status"] == "failed").sum()
    )
)

print(
    "Animals processed:",
    log_df.loc[
        log_df["status"] == "processed",
        "subject"
    ].nunique()
)

print("Processing log:", LOG_PATH)


In [ ]:

# ============================================================
# CELL 5 — CNTNAP2 ANIMAL-LEVEL AGGREGATION
# ============================================================

GENOTYPE = "Cntnap2"

BAND_COLUMNS = [
    f"{band}_{metric}"
    for band in BANDS
    for metric in (
        "absolute_uv2",
        "relative_pct"
    )
]

# ------------------------------------------------------------
# 1. Collect all completed insertion files
# ------------------------------------------------------------

result_files = sorted(
    path
    for path in INSERTION_DIR.glob("*.csv")
    if not path.name.endswith("_windows.csv")
)

if not result_files:
    raise RuntimeError(
        "No insertion-level results found"
    )

insertion_results = pd.concat(
    [
        pd.read_csv(path)
        for path in result_files
    ],
    ignore_index=True
)

# Only include the current verified Cntnap2 inventory.
valid_pids = set(
    inventory["pid"].astype(str)
)

insertion_results = insertion_results.loc[
    (
        insertion_results["genotype"] == GENOTYPE
    )
    &
    (
        insertion_results["pid"].astype(str).isin(
            valid_pids
        )
    )
].copy()

if insertion_results.empty:
    raise RuntimeError(
        "No valid Cntnap2 insertion results"
    )

# Verify numeric band-power values.
if insertion_results[
    BAND_COLUMNS
].isna().any().any():
    raise RuntimeError(
        "Missing band-power values in saved results"
    )

# Save insertion-level table.
insertion_results.to_csv(
    ROOT / "Cntnap2_LFP_insertion_results.csv",
    index=False
)

# ------------------------------------------------------------
# 2. Aggregate repeated insertions within each animal
# ------------------------------------------------------------

animal_results = (
    insertion_results
    .groupby(
        [
            "subject",
            "genotype",
            "region",
            "subregion"
        ],
        as_index=False
    )
    .agg(
        **{
            col: (col, "median")
            for col in BAND_COLUMNS
        },
        n_sessions=(
            "eid", "nunique"
        ),
        n_insertions=(
            "pid", "nunique"
        ),
        total_channels=(
            "n_channels", "sum"
        )
    )
)

# ------------------------------------------------------------
# 3. Determine animal coverage
# ------------------------------------------------------------

coverage = (
    animal_results
    .groupby(
        ["region", "subregion"],
        as_index=False
    )
    .agg(
        n_animals=(
            "subject", "nunique"
        )
    )
)

coverage["status"] = np.where(
    coverage["n_animals"] >= MIN_ANIMALS,
    "INCLUDED",
    "INSUFFICIENT_DATA"
)

# ------------------------------------------------------------
# 4. Keep groups with at least 5 animals
# ------------------------------------------------------------

eligible = coverage.loc[
    coverage["status"] == "INCLUDED",
    ["region", "subregion"]
]

final_results = animal_results.merge(
    eligible,
    on=["region", "subregion"],
    how="inner"
)

# ------------------------------------------------------------
# 5. Group-level descriptive summary
# ------------------------------------------------------------

summary = (
    final_results
    .groupby(
        ["region", "subregion"],
        as_index=False
    )
    .agg(
        n_animals=(
            "subject", "nunique"
        ),
        **{
            col: (col, "median")
            for col in BAND_COLUMNS
        }
    )
)

# ------------------------------------------------------------
# 6. Save all tables
# ------------------------------------------------------------

animal_results.to_csv(
    ROOT / "Cntnap2_LFP_all_animal_results.csv",
    index=False
)

coverage.to_csv(
    ROOT / "Cntnap2_LFP_region_eligibility.csv",
    index=False
)

final_results.to_csv(
    ROOT / "Cntnap2_LFP_final_animal_results.csv",
    index=False
)

summary.to_csv(
    ROOT / "Cntnap2_LFP_group_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("=" * 60)
print("CNTNAP2 LFP PROCESSING COMPLETE")
print("=" * 60)

print(
    "Animals analyzed:",
    animal_results["subject"].nunique()
)

print(
    "Eligible region/subregion groups:",
    len(eligible)
)

print(
    "Final animal-level rows:",
    len(final_results)
)

print("\nREGION COVERAGE:")
display(
    coverage.sort_values(
        ["region", "subregion"]
    )
)

print("\nGROUP SUMMARY:")
display(summary)

print("\nSaved to:", ROOT)
